# Task 3 – Running a model locally & optimisation

**Goal:** run the same small model (Qwen2.5-1.5B-Instruct) in two settings and compare them:

| Setting | What it means |
|---|---|
| **FP16** | every weight stored as a 16-bit number (2 bytes) – the "full quality" version |
| **Q4_K_M** | weights compressed to roughly 4–5 bits each – about 3–4× smaller |

We use **Ollama**, a tool that downloads and runs models with one command.

**Before you start:** on Kaggle: Settings → Accelerator → **GPU T4 x2**, and Settings → **Internet on**. (On Colab: Runtime → Change runtime type → T4 GPU.) Then run each cell in order (Shift+Enter).

## Step 1 – Check we have a GPU
You should see a table that mentions **Tesla T4** with about **15 GB** of memory (Kaggle shows two of them – we only use the first). If you get an error, go back and change the runtime type.

In [ ]:
!nvidia-smi

## Step 2 – Install Ollama
This downloads Ollama into the Colab machine (takes ~1 minute). `zstd` is a small helper Ollama's installer needs.

In [ ]:
!(command -v sudo >/dev/null && SUDO=sudo || SUDO=; $SUDO apt-get -qq update > /dev/null && $SUDO apt-get -qq install -y zstd pciutils > /dev/null)
!curl -fsSL https://ollama.com/install.sh | sh

## Step 3 – Start the Ollama server
Ollama works as a small **server** running in the background. Our Python code sends it prompts over `http://localhost:11434` and gets answers back.

`subprocess.Popen` starts it in the background; the loop just waits until it is ready.

In [ ]:
import subprocess, time, requests, os

URL = "http://localhost:11434"
# Kaggle gives 2 GPUs; we use only GPU 0 so the memory numbers are simple and fair.
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        requests.get(URL)
        print("Ollama server is running")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

## Step 4 – Download the two versions of the model
Same model, two precisions. Watch the download sizes – that alone shows what quantisation does.

In [ ]:
MODELS = {
    "FP16":   "qwen2.5:1.5b-instruct-fp16",
    "Q4_K_M": "qwen2.5:1.5b-instruct-q4_K_M",
}
for tag in MODELS.values():
    !ollama pull {tag}

!ollama list   # shows size on disk of each version

## Step 5 – The 10 test prompts
They cover different skills on purpose – facts, maths, reasoning, code, JSON formatting, summarising, translation, creative writing and instruction-following – because quantisation usually hurts some skills (maths, exact formats) more than others.

*You can change these to your own prompts – it is actually better if some are yours.*

In [ ]:
PROMPTS = [
    "What is the capital of Australia? Answer in one sentence.",
    "A shop sells pens at Rs 12 each. If I buy 7 pens and pay with a Rs 100 note, how much change do I get? Show your working.",
    "If all bloops are razzies and all razzies are lazzies, are all bloops definitely lazzies? Explain briefly.",
    "Write a Python function that checks whether a string is a palindrome, ignoring spaces and capital letters.",
    "Convert this into JSON with keys name, age, city: 'Riya is 23 years old and lives in Pune.' Output only the JSON.",
    "Summarise in two sentences: Photosynthesis is the process by which green plants use sunlight, water and carbon dioxide to make glucose and release oxygen. It happens mainly in the leaves, inside chloroplasts, which contain the green pigment chlorophyll.",
    "Translate into Hindi: 'The meeting has been moved to tomorrow morning.'",
    "Write a four-line poem about monsoon rain in a city.",
    "List exactly three advantages of remote work. Use a numbered list and nothing else.",
    "What is 17 multiplied by 23? Give only the number.",
]
print(len(PROMPTS), "prompts ready")

## Step 6 – Helper functions to measure speed and memory

**How speed is measured:** for each answer Ollama reports
- `eval_count` = how many tokens it generated
- `eval_duration` = how long that took, in nanoseconds

So **tokens per second = eval_count ÷ (eval_duration / 1,000,000,000)**.

**How memory is measured (two ways):**
1. `nvidia-smi` – total GPU memory in use, read before and after the model is loaded. The difference is what the model costs.
2. Ollama's `/api/ps` endpoint – Ollama's own report of how much GPU memory (VRAM) the loaded model takes.

**Fair-comparison settings:** `temperature = 0` (no randomness, so the same prompt always gives the same answer), same `seed`, same context size (`num_ctx = 2048`) and same max answer length (`num_predict = 300`) for both models.

In [ ]:
def gpu_mem_used_mb():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True).stdout
    return int(out.strip().splitlines()[0])

OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 2048, "num_predict": 300}

def unload(tag):
    """keep_alive=0 tells Ollama to remove the model from GPU memory."""
    requests.post(f"{URL}/api/generate", json={"model": tag, "keep_alive": 0})
    time.sleep(3)

def benchmark(name, tag):
    base_mem = gpu_mem_used_mb()

    # Warm-up: the first call also loads the model into GPU memory,
    # so we do NOT time it (it would make the speed look worse than it is).
    requests.post(f"{URL}/api/generate",
                  json={"model": tag, "prompt": "Hi", "stream": False, "options": OPTIONS})

    loaded_mem = gpu_mem_used_mb()
    ps = requests.get(f"{URL}/api/ps").json()["models"]
    vram_mb = next(m["size_vram"] for m in ps if m["name"] == tag) / 1024**2

    rows = []
    for i, prompt in enumerate(PROMPTS, 1):
        r = requests.post(f"{URL}/api/generate",
                          json={"model": tag, "prompt": prompt,
                                "stream": False, "options": OPTIONS}).json()
        tps = r["eval_count"] / (r["eval_duration"] / 1e9)
        rows.append({"setting": name, "prompt_id": i, "prompt": prompt,
                     "tokens_generated": r["eval_count"],
                     "tokens_per_sec": round(tps, 1),
                     "response": r["response"].strip()})
        print(f"{name} | prompt {i:2d} | {r['eval_count']:3d} tokens | {tps:6.1f} tok/s")

    unload(tag)
    memory = {"setting": name,
              "gpu_mem_before_MB": base_mem,
              "gpu_mem_after_load_MB": loaded_mem,
              "model_cost_nvidia_smi_MB": loaded_mem - base_mem,
              "ollama_reported_vram_MB": round(vram_mb)}
    return rows, memory

## Step 7 – Run the benchmark on both settings
This takes a few minutes. You will see one line per prompt.

In [ ]:
all_rows, all_memory = [], []
for name, tag in MODELS.items():
    rows, mem = benchmark(name, tag)
    all_rows += rows
    all_memory.append(mem)
    print()

## Step 8 – Summary table (these are YOUR numbers for the report)

In [ ]:
import pandas as pd

results = pd.DataFrame(all_rows)
memory  = pd.DataFrame(all_memory).set_index("setting")

speed = results.groupby("setting")["tokens_per_sec"].agg(["mean", "min", "max"]).round(1)
speed.columns = ["avg_tok_per_s", "min_tok_per_s", "max_tok_per_s"]

summary = speed.join(memory)
summary

## Step 9 – Compare the answers side by side
Read each pair carefully. Note where Q4 is **worse**, **the same**, or even **different-but-fine**.

In [ ]:
from IPython.display import display, Markdown

for i in range(1, len(PROMPTS) + 1):
    fp = results[(results.setting == "FP16")   & (results.prompt_id == i)].response.iloc[0]
    q4 = results[(results.setting == "Q4_K_M") & (results.prompt_id == i)].response.iloc[0]
    display(Markdown(f"---\n### Prompt {i}: {PROMPTS[i-1]}\n"
                     f"**FP16:**\n\n{fp}\n\n**Q4_K_M:**\n\n{q4}"))

## Step 10 – Save results (upload these to the `task3_local_inference` folder on GitHub)

In [ ]:
results.to_csv("task3_results.csv", index=False)
summary.to_csv("task3_summary.csv")
!cat ollama.log | grep -i "offload" | head -5   # shows how many layers went to GPU

try:   # on Colab: download straight to your computer
    from google.colab import files
    files.download("task3_results.csv"); files.download("task3_summary.csv")
except ImportError:   # on Kaggle: the files appear in the Output panel on the right
    print("Saved. On Kaggle, find the CSV files under Output (right side panel) and download them.")

## Step 11 – Your observations (write these yourself after reading Step 9)

For each prompt, mark: **Same** / **Q4 slightly worse** / **Q4 clearly worse** / **Q4 better**.

| # | Skill | Verdict | One-line note |
|---|---|---|---|
| 1 | Fact | | |
| 2 | Maths word problem | | |
| 3 | Logic | | |
| 4 | Code | | |
| 5 | JSON format | | |
| 6 | Summary | | |
| 7 | Hindi translation | | |
| 8 | Poem | | |
| 9 | Follow format exactly | | |
| 10 | Exact arithmetic | | |

**Overall:** (2–3 sentences in your own words – how much faster/smaller was Q4, and was the quality loss worth it?)